# MedGemma CT Chat — Colab Backend

This notebook runs the MedGemma 1.5 4B FastAPI backend on a free Colab T4 GPU.
Your local frontend connects to it via a Cloudflare tunnel.

**Prerequisites:**
1. A HuggingFace account with access to [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it) (click "Agree and access")
2. A HuggingFace token from [settings/tokens](https://huggingface.co/settings/tokens)
3. Add the token as a Colab secret: click the key icon in the left sidebar, add `HF_TOKEN`

**Usage:** Runtime > Change runtime type > T4 GPU, then Run All (Ctrl+F9)

## 1. Install dependencies

In [ ]:
!pip install -q fastapi uvicorn[standard] python-multipart pydicom "numpy<2.1" Pillow accelerate huggingface_hub scipy hydra-core iopath
!pip install -q transformers@git+https://github.com/huggingface/transformers.git

# Install MedSAM2 for lesion segmentation
!rm -rf /content/MedSAM2
!git clone https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
!cd /content/MedSAM2 && pip install -q -e .

# Download the CT lesion checkpoint from HuggingFace
from huggingface_hub import hf_hub_download
medsam2_ckpt = hf_hub_download(repo_id="wanglab/MedSAM2", filename="MedSAM2_CTLesion.pt")
print(f"MedSAM2 checkpoint: {medsam2_ckpt}")

## 2. Authenticate with HuggingFace

In [ ]:
import os

# Try Colab secrets first, fall back to manual login
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    print("Colab secrets not available. Using manual login:")
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Clone repo & verify GPU

In [ ]:
import torch
import os

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

# Clone the repository
!rm -rf /content/medgemma
!git clone https://github.com/mnadiri/medgemma.git /content/medgemma
!cd /content/medgemma && git checkout claude/dicom-ct-ai-chat-BDapi

# Set MedSAM2 checkpoint path for the backend
from huggingface_hub import hf_hub_download
os.environ["MEDSAM2_CHECKPOINT"] = hf_hub_download(
    repo_id="wanglab/MedSAM2", filename="MedSAM2_CTLesion.pt"
)
print(f"\nMedSAM2 checkpoint: {os.environ['MEDSAM2_CHECKPOINT']}")

# Add MedSAM2 to Python path so the backend can import sam2
# (Hydra resolves config via pkg://sam2 automatically)
import sys
sys.path.insert(0, "/content/MedSAM2")

print("Repo cloned and checked out. MedSAM2 configured.")

## 4. Start the FastAPI backend

In [ ]:
import subprocess
import sys
import time
import os

# Ensure MedSAM2 is on the path for the subprocess
env = os.environ.copy()
pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = f"/content/MedSAM2:{pythonpath}"

# Start backend in background
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8001"],
    cwd="/content/medgemma/web/backend",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Backend starting... (loading MedGemma + MedSAM2, this takes ~2-3 minutes)")

# Wait for model to load by polling health endpoint
import urllib.request
import json

for i in range(180):  # Wait up to 3 minutes
    time.sleep(2)
    try:
        resp = urllib.request.urlopen("http://localhost:8001/api/health", timeout=2)
        data = json.loads(resp.read())
        if data.get("model_loaded"):
            medsam_status = "loaded" if data.get("medsam2_loaded") else "disabled"
            print(f"\nBackend ready! (took ~{(i+1)*2}s)")
            print(f"  MedGemma: loaded ({data.get('backend')} mode)")
            print(f"  MedSAM2:  {medsam_status}")
            break
    except Exception:
        if i % 10 == 0:
            print(f"  Still loading... ({(i+1)*2}s)")
else:
    print("WARNING: Backend may not have started. Check logs below:")
    backend_proc.terminate()
    print(backend_proc.stdout.read())

## 5. Create public tunnel (Cloudflare)

In [ ]:
import subprocess
import re
import time

# Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Start tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for tunnel URL
tunnel_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
    if match:
        tunnel_url = match.group(1)
        break

if tunnel_url:
    print("=" * 60)
    print(f"TUNNEL URL: {tunnel_url}")
    print(f"API URL:    {tunnel_url}/api")
    print("=" * 60)
    print()
    print("On your Mac, start the frontend with:")
    print(f'  cd ~/medgemma/web/frontend')
    print(f'  VITE_BACKEND_URL={tunnel_url}/api npm run dev')
    print()
    print("Or create web/frontend/.env.local with:")
    print(f'  VITE_BACKEND_URL={tunnel_url}/api')
else:
    print("ERROR: Could not get tunnel URL. Try re-running this cell.")

## 6. Keep alive & monitor

Run this cell to keep the notebook alive and see backend logs. Stop it (click stop button) when you're done.

In [ ]:
import time

print("Backend is running. Logs will appear below.")
print("Press the stop button to shut down.\n")

try:
    while True:
        line = backend_proc.stdout.readline()
        if line:
            print(line, end="")
        elif backend_proc.poll() is not None:
            print("\nBackend process exited!")
            break
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("\nStopping...")
    backend_proc.terminate()
    tunnel_proc.terminate()
    print("Backend and tunnel stopped.")